In [ ]:
import sys
from pathlib import Path
_ROOT = Path.cwd().parent.resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from    pathlib         import  Path
from    tqdm.notebook   import  tqdm
from    time            import  perf_counter

import  torch
from    torch.utils.data    import  TensorDataset, DataLoader
import  matplotlib.pyplot   as  plt

from    core.utils    import  velocity_grid, relative_error
from    core.numerical.math   import  area_of_unit_sphere
from    core.numerical.solver     import  LAMBDA, one_step_RK4_classic
from    core.numerical.distribution   import  maxwellian_homogeneous
from    core.neural.collision_op  import  PRFNO


from    loader  import  load_dataset


DEVICE = torch.device('cuda:0')
base_path = _ROOT
assert base_path.exists()

In [ ]:
DIMENSION   = 2
N_LAYERS    = 4
MAX_FREQ    = 8
SEED        = 0
MODEL_TYPE  = "prfno"

RESOLUTION  = 2**10
MAX_V       = 3.0/LAMBDA

PART_INIT   = 1
COL_COEFF   = 1/area_of_unit_sphere(DIMENSION)
EXP_SPEED   = torch.arange(1)*0.0

BATCH_SIZE  = 1

path = base_path/f"pretrained"
model_and_history = torch.load(path/f"{MODEL_TYPE.upper()}_{DIMENSION}D"/"model_and_history.pth", weights_only=False, map_location='cpu')
print(model_and_history.keys())

dict_keys(['args', 'state_dict', 'train_history'])


In [ ]:
v_grid = velocity_grid(DIMENSION, RESOLUTION, MAX_V)
f_init = maxwellian_homogeneous(v_grid, torch.ones(1, 1), torch.zeros(1, DIMENSION), torch.ones(1, 1))
f_init = f_init.reshape(1, *v_grid.shape[:-1], 1)
param = EXP_SPEED*torch.ones(1, 1)

# Generate dataloader
test_dataloader = DataLoader(TensorDataset(f_init, param), batch_size=BATCH_SIZE, shuffle=False)

In [4]:
prfno = PRFNO(**model_and_history['args']).to(DEVICE)
prfno.load_state_dict(model_and_history['state_dict'])
prfno.eval()
v_grid = velocity_grid(DIMENSION, RESOLUTION, MAX_V, device=DEVICE)

In [5]:
torch.cuda.reset_peak_memory_stats(DEVICE)
torch.cuda.synchronize(DEVICE)

elapse_time = perf_counter()
with torch.inference_mode():
    for f_init, params in test_dataloader:
        f_pred = f_init.to(DEVICE)
        params = params.to(DEVICE)
        q_pred = COL_COEFF*prfno.forward(f_pred, v_grid, params)
elapse_time = perf_counter() - elapse_time

torch.cuda.synchronize(DEVICE)
peak_memory_fno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
print(f"Elapsed time: {elapse_time:.4e} seconds")
print(f"Peak memory usage: {peak_memory_fno:.2f} MB")

Elapsed time: 2.4380e-01 seconds
Peak memory usage: 2149.74 MB


In [6]:
%%timeit -r 30 -n 5
with torch.inference_mode():
    for f_init, params in test_dataloader:
        f_pred = f_init.to(DEVICE)
        params = params.to(DEVICE)
        q_pred = COL_COEFF*prfno.forward(f_pred, v_grid, params)

89.9 ms ± 3.1 ms per loop (mean ± std. dev. of 30 runs, 5 loops each)


End of file